# [LAB] 2-3 실습

(작성 예정)

**목차**

- 1. 실습 방법
- 2. app (예제) (과제)
  - 2.1 RAG 파이프라인 초기화 (앱 시작 시 한 번만 실행) (예제)
  - 2.2 Streamlit UI (과제)
- 3. test
  - 3.1 AI에게 물어보기
- 4. 정리

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))  # 프로젝트의 .env 파일을 읽습니다

# 값은 출력하지 않습니다. 빠진 설정이 있는지만 봅니다.
provider = os.getenv("LLM_PROVIDER", "samsung").strip().lower()
필요한_이름 = {
    "openai": ["OPENAI_API_KEY", "OPENAI_MODEL"],
    "samsung": ["api_base_url", "model", "credential_key", "send_system_name", "user_id"],
}.get(provider, ["LLM_PROVIDER"])
빠진_이름 = [name for name in 필요한_이름 if not os.getenv(name)]

import config.llm  # 실습 코드가 쓰는 설정 모듈을 불러옵니다

if 빠진_이름:
    print(f"준비가 덜 되었습니다. 빠진 설정 {len(빠진_이름)}개가 있습니다. 강사에게 알려 주세요.")
    print("(강사 확인용:", ", ".join(빠진_이름), ")")
else:
    print("준비 완료: LLM에 연결할 설정이 모두 있습니다.")

**볼 지점**
- "준비 완료"가 나오면 다음으로 넘어갑니다.
- "준비가 덜 되었습니다"가 나오면 강사에게 알립니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 강사에게 알립니다.

## 1. 실습 방법

실습은 두 종류입니다. 제목 옆 표시와 코드 주석으로 구분합니다.

- **예제** `# ✍️ CHANGE_HERE:` 바로 아래 줄의 값을 바꿔 다시 실행해 봅니다.
- **과제** `# ✍️ WRITE_HERE:` 아래 빈칸(`...`)을 AI와 함께 채웁니다. **프롬프트 힌트**를 보고 지시를 직접 씁니다.

실행한 뒤에는 **성공 기준**과 비교해 스스로 판단합니다. 🟢는 되어야 하는 것, ⛔는 다시 해야 하는 신호입니다. 과제는 그다음 접힌 "정답 보기"와 비교합니다.

## 2. app <small>(예제) (과제)</small>

(작성 예정)

준비 코드입니다. 필요한 도구를 불러오고 모델을 준비합니다. 접혀 있으니 펼치지 않아도 되고, 실행만 합니다.

In [ ]:
from pathlib import Path

import streamlit as st
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_community.document_loaders import PDFMinerLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter

from config.embeddings import get_embeddings
from config.llm import get_llm

load_dotenv()

### 2.1 RAG 파이프라인 초기화 (앱 시작 시 한 번만 실행) <small>(예제)</small>

In [ ]:
# ── RAG 파이프라인 초기화 (앱 시작 시 한 번만 실행) ─────────────────────────
# 사용자가 어떤 질문을 한다 -> agent를 `호출`한다. 
    # 결국 가우스 모델을 호출할건데 -> 함수 실행 결과에 불과 하겠죠.
@st.cache_resource
def load_rag_chain():
    PDF_PATH = Path(__file__).parent / 'data' / 'meritz_pet_insurance.pdf'

    loader = PDFMinerLoader(str(PDF_PATH.resolve()))
    documents = loader.load()

    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=75)
    chunks = splitter.split_documents(documents)

    embeddings = get_embeddings()
    vectorstore = Chroma.from_documents(chunks, embedding=embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

    llm = get_llm()
    prompt = ChatPromptTemplate.from_messages([
        ('system', '당신은 문서 기반 QA 어시스턴트입니다. 주어진 context만 근거로 답하세요.\n\ncontext:\n{context}'),
        ('human', '{question}'),
    ])

    def format_docs(docs):
        return '\n\n'.join(doc.page_content for doc in docs)

    chain = (
        {'context': retriever | format_docs, 'question': RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return chain

### 2.2 Streamlit UI <small>(과제)</small>

**✍️ WRITE_HERE: 사용자 메시지를 session_state에 저장해 주세요.**

(작성 예정)

**프롬프트 힌트** (작성 예정)

**성공 기준**

**✍️ WRITE_HERE: RAG 체인의 stream() 결과를 Streamlit에 출력해 주세요.**

(작성 예정)

**프롬프트 힌트** (작성 예정)

**성공 기준**

**✍️ WRITE_HERE: AI 응답을 session_state에 저장해 주세요.**

(작성 예정)

**프롬프트 힌트** (작성 예정)

**성공 기준**

In [ ]:
# ── Streamlit UI ──────────────────────────────────────────────────────────────

st.title('보험 약관 QA 챗봇')
st.caption('메리츠 펫보험 약관을 기반으로 답변합니다.')

# 대화 기록을 session_state에 저장 (새로고침 전까지 유지)
    # 어디까지나 화면에 보여주기 위한 session 관리인 것
    # 이 message를 토대로 LLM이 이전 대화를 기억한다거나 그런건 X
if 'messages' not in st.session_state:
    st.session_state.messages = []

# 이전 대화 표시
for msg in st.session_state.messages:
    # msg -> LLM이 만들어낸 답변 messages (우리가 아까까지 graph의 State에 담던 그 messages)
        # user냐? AI냐?에 따라서
        # 대화창을 어디에 보여줄것인지를 달리 할 것
    with st.chat_message(msg['role']):
        st.write(msg['content'])

# 사용자 입력
if prompt := st.chat_input('질문을 입력하세요 (예: 보험료는 어떻게 계산되나요?)'):
    # 사용자 메시지 표시
    # ✍️ WRITE_HERE: 사용자 메시지를 session_state에 저장해 주세요.
    ...  # ✍️ 여기를 채웁니다
    with st.chat_message('user'):
        st.write(prompt)

    # AI 응답 (스트리밍)
    with st.chat_message('assistant'):
        # 한개의 runnable을 받았다.
        rag_chain = load_rag_chain()
        # ✍️ WRITE_HERE: RAG 체인의 stream() 결과를 Streamlit에 출력해 주세요.
            # runnable을 실행하는 방법 -> invoke, stream
        ...  # ✍️ 여기를 채웁니다
    # ✍️ WRITE_HERE: AI 응답을 session_state에 저장해 주세요.
    ...  # ✍️ 여기를 채웁니다

<details><summary>정답 보기</summary>

```python
st.session_state.messages.append({'role': 'user', 'content': prompt})
```

```python
response = st.write_stream(rag_chain.stream(prompt))
```

```python
st.session_state.messages.append({'role': 'assistant', 'content': response})
```

</details>

(작성 예정)

## 3. test

(작성 예정)

준비 코드입니다. 필요한 도구를 불러오고 모델을 준비합니다. 접혀 있으니 펼치지 않아도 되고, 실행만 합니다.

In [ ]:
import streamlit as st

st.title("버튼 누르면 스크립트 전체 다시 실행")

if "clicks" not in st.session_state:  
    st.session_state.clicks = 0

if st.button(' 클릭 '):
    st.session_state.clicks +=1
    st.balloons()

st.write(f"풍선 발사 {st.session_state.clicks}회")

(작성 예정)

### 3.1 AI에게 물어보기

**프롬프트 힌트** (작성 예정)

**성공 기준**

- 🟢 (작성 예정)

## 4. 정리

(작성 예정)